# Orivox - Separador de Pistas (gratuito)

Separa sua musica em pistas individuais e gera extras para o Orivox.

### Pistas que voce recebe
Voz principal - Backing vocals - Bateria - Baixo - Guitarra - Piano - Outros

### Extras (opcionais)
Mapa de batidas (metronomo) - Cifra dos acordes - Partitura da voz

### Como usar
1. Menu **Ambiente de execucao > Alterar tipo de ambiente > T4 GPU** e salve.
2. Rode as celulas na ordem, clicando no play de cada uma.
3. Na celula 2, envie sua musica quando o botao aparecer.
4. A celula 5 baixa o ZIP com as pistas. As celulas 6+ sao extras opcionais.

*IMPORTANTE: as pistas (celulas 1 a 5) rodam primeiro e ficam prontas. Os extras
(6 em diante) instalam ferramentas mais pesadas so quando voce os roda, para NAO
atrapalhar a separacao das pistas.*

## Celula 1 - Instalar as ferramentas da separacao (rode uma vez por sessao)

In [ ]:
%%capture
# So o essencial para SEPARAR as pistas - sem pacotes pesados que dao conflito.
# Os extras (partitura, acordes) instalam suas ferramentas depois, nas celulas 6+.
!pip install -U demucs
!pip install "audio-separator[cpu]"


In [ ]:
print('Ferramentas da separacao instaladas! Siga para a celula 2.')

## Celula 2 - Enviar sua musica (MP3, WAV, M4A...)

In [ ]:
from google.colab import files
import os, shutil

# Aviso amigavel sobre a GPU (nao impede, so avisa).
try:
    import torch
    if torch.cuda.is_available():
        print('GPU ligada:', torch.cuda.get_device_name(0), '- otimo, vai ser rapido.')
    else:
        print('ATENCAO: GPU nao esta ligada. Funciona, mas mais devagar.')
        print('Para ligar: Ambiente de execucao > Alterar tipo > T4 GPU > Salvar. Depois rode de novo.')
except Exception:
    pass

if os.path.exists('/content/entrada'):
    shutil.rmtree('/content/entrada')
os.makedirs('/content/entrada', exist_ok=True)

print('\nClique em "Escolher arquivos" e selecione sua musica:')
enviados = files.upload()

for nome in enviados:
    shutil.move(nome, f'/content/entrada/{nome}')
    print(f'Recebido: {nome}')
arquivo = list(enviados.keys())[0]

## Celula 3 - Separar os instrumentos e a voz (Demucs, 6 pistas)

- `htdemucs_6s` gera 6 pistas: voz, bateria, baixo, guitarra, piano e outros

In [ ]:
MODO = 'htdemucs_6s'

import subprocess, os, shutil, glob
if os.path.exists('/content/saida'):
    shutil.rmtree('/content/saida')
entrada = f'/content/entrada/{arquivo}'
print(f'Separando "{arquivo}" em 6 pistas... aguarde (pode levar alguns minutos).')

proc = subprocess.run(
    ['demucs', '-n', MODO, '--mp3', '--mp3-bitrate', '320', '-o', '/content/saida', entrada],
    capture_output=True, text=True
)

# VERIFICACAO CRITICA: confere se as 6 pistas foram MESMO geradas.
# Se o demucs falhar, TRAVA aqui com dica clara - em vez de seguir e sobrar so a voz.
pistas = glob.glob('/content/saida/*/*/*.mp3')
esperadas = {'drums','bass','other','vocals','guitar','piano'}
achadas = set(os.path.splitext(os.path.basename(p))[0] for p in pistas)

if proc.returncode != 0 or not esperadas.issubset(achadas):
    print('=== A separacao em 6 pistas NAO foi concluida. ===')
    print('Pistas geradas:', sorted(achadas) if achadas else 'NENHUMA')
    print('Faltaram:', sorted(esperadas - achadas))
    print()
    print('--- Detalhes (ultimas linhas) ---')
    print((proc.stderr or proc.stdout or '')[-2000:])
    print()
    print('DICAS: 1) Ligue a GPU (Ambiente de execucao > T4 GPU) e rode de novo.')
    print('       2) Se faltou memoria, teste com uma musica mais curta.')
    print('       3) Se rodou algum extra (celula 6+) antes, reinicie o ambiente')
    print('          (Ambiente de execucao > Reiniciar) e rode da celula 1 de novo.')
    raise SystemExit('Separacao incompleta - veja as dicas acima.')

print('Pronto! As 6 pistas foram geradas:')
for p in sorted(pistas):
    print('  -', os.path.basename(p))

## Celula 4 - Dividir a voz em PRINCIPAL e BACKING VOCALS

Pega a pista de voz da celula 3 e isola a voz principal dos vocais de apoio.

In [ ]:
import glob, os, shutil
from audio_separator.separator import Separator

candidatos = glob.glob('/content/saida/*/*/vocals.*')
if not candidatos:
    raise SystemExit('Pista de voz nao encontrada. Rode a celula 3 primeiro (ela precisa terminar sem erro).')
voz_completa = candidatos[0]
print('Pista de voz localizada:', os.path.basename(voz_completa))

if os.path.exists('/content/vozes'):
    shutil.rmtree('/content/vozes')
os.makedirs('/content/vozes', exist_ok=True)

print('Separando voz principal dos backing vocals... aguarde.')
sep = Separator(output_dir='/content/vozes', output_format='mp3')
sep.load_model(model_filename='5_HP-Karaoke-UVR.pth')
saidas = sep.separate(voz_completa)

print('Pronto! Arquivos de voz gerados:')
for s in saidas:
    print('  -', os.path.basename(s))

## Celula 5 - Baixar as pistas (o essencial ja fica pronto aqui!)

Junta as 7 pistas e baixa o ZIP. **Se voce so quer as pistas, pode parar aqui.**
As celulas seguintes (6, 7, 8) sao extras opcionais.

In [ ]:
import shutil, os, glob
from google.colab import files

try:
    arquivo
except NameError:
    arquivo = os.path.basename(glob.glob('/content/entrada/*')[0])
base = os.path.splitext(arquivo)[0]
final = f'/content/{base}_Orivox'
if os.path.exists(final):
    shutil.rmtree(final)
os.makedirs(final, exist_ok=True)

# 1) Pistas do Demucs (a voz completa entra como referencia).
nomes_pt = {'drums':'Bateria','bass':'Baixo','other':'Outros','guitar':'Guitarra','piano':'Piano'}
pastas = [p for p in glob.glob('/content/saida/*/*') if os.path.isdir(p)]
if pastas:
    for f in glob.glob(f'{pastas[0]}/*'):
        raiz = os.path.splitext(os.path.basename(f))[0]
        ext = os.path.splitext(f)[1]
        if raiz == 'vocals':
            shutil.copy(f, f'{final}/Vozes juntas (referencia) - {base}{ext}')
        elif raiz in nomes_pt:
            shutil.copy(f, f'{final}/{nomes_pt[raiz]} - {base}{ext}')

# 2) Voz principal e backing vocals.
for f in glob.glob('/content/vozes/*'):
    nome = os.path.basename(f); ext = os.path.splitext(f)[1]
    if '(Vocals)' in nome:
        shutil.copy(f, f'{final}/Voz principal - {base}{ext}')
    elif '(Instrumental)' in nome:
        shutil.copy(f, f'{final}/Backing vocals - {base}{ext}')

# Confere quantas pistas de fato entraram
pistas_final = [f for f in os.listdir(final) if os.path.isfile(os.path.join(final,f))]
print(f'Pistas reunidas ({len(pistas_final)}):')
for f in sorted(pistas_final):
    print('  -', f)

if len(pistas_final) < 6:
    print('\nATENCAO: esperava 7 pistas mas achei', len(pistas_final), '.')
    print('Confira se as celulas 3 e 4 terminaram sem erro.')

zipado = shutil.make_archive(f'/content/{base}_Orivox_pistas', 'zip', final)
files.download(zipado)
print('\nDownload das pistas iniciado! Importe no console multipista do Orivox.')

## Celula 6 - EXTRA: Mapa de batidas (para o metronomo do Orivox)

Gera um `_batidas.json` com cada batida e tempo forte da musica. Importe no
Metronomo Inteligente do Orivox para o clique seguir a musica com precisao.
*Opcional - so rode se quiser o mapa.*

In [ ]:
# Instala o librosa SO AGORA (isolado), para nao atrapalhar a separacao das pistas.
import subprocess
subprocess.run(['pip','install','-q','librosa','soundfile'], check=False)

import json, os, glob
import numpy as np
from google.colab import files

entrada_lista = glob.glob('/content/entrada/*')
if not entrada_lista:
    raise SystemExit('Envie a musica na celula 2 primeiro.')
caminho = entrada_lista[0]
base = os.path.splitext(os.path.basename(caminho))[0]

import librosa
print('Analisando as batidas (librosa)... 1 a 2 minutos.')
y, sr = librosa.load(caminho, sr=22050, mono=True)
tempo, frames = librosa.beat.beat_track(y=y, sr=sr, trim=False)
beats = librosa.frames_to_time(frames, sr=sr).tolist()
env = librosa.onset.onset_strength(y=y, sr=sr)
forca = [float(env[min(f, len(env)-1)]) for f in frames]
melhor_o, melhor_s = 0, -1
for o in range(4):
    s = sum(forca[o::4])
    if s > melhor_s: melhor_s, melhor_o = s, o
numeros = [((i - melhor_o) % 4) + 1 for i in range(len(beats))]

iv = np.diff(beats)
bpm = round(float(60/np.median(iv)), 1) if len(iv) else 0
mapa = {'origem':'librosa', 'bpm':bpm, 'beats':[round(b,4) for b in beats], 'beat_numbers':numeros}
os.makedirs('/content/extras', exist_ok=True)
saida = f'/content/extras/{base}_batidas.json'
json.dump(mapa, open(saida,'w'))
print(f'Mapa gerado: {len(beats)} batidas, ~{bpm} BPM.')
files.download(saida)
print('Importe no Metronomo Inteligente do Orivox.')

## Celula 7 - EXTRA: Partitura de referencia da voz

Transcreve a voz principal em MIDI (basic-pitch do Spotify) e gera um MusicXML.
Importe o `_voz.mid` na Partitura do Orivox (botao Importar referencia).
*Opcional. Roda por ultimo porque instala ferramentas mais pesadas - por isso as
pistas ja foram baixadas na celula 5, sem risco.*

In [ ]:
# Instala o basic-pitch SO AGORA (isolado, apos as pistas ja baixadas).
# A versao 'onnx' evita o TensorFlow pesado que conflitava com o demucs.
# Fixamos setuptools antes, para evitar erros de build em Python novo.
import subprocess
subprocess.run(['pip','install','-q','setuptools<81','wheel'], check=False)
subprocess.run(['pip','install','-q','basic-pitch[onnx]','music21'], check=False)

import glob, os
# Usa a VOZ PRINCIPAL isolada (melhor fonte). Se nao houver, usa a voz completa.
voz_ref = None
for padrao in ['/content/vozes/*(Vocals)*', '/content/saida/*/*/vocals.*']:
    achados = glob.glob(padrao)
    if achados:
        voz_ref = achados[0]; break
if not voz_ref:
    raise SystemExit('Voz nao encontrada. Rode as celulas 3 e 4 primeiro.')
print('Transcrevendo a voz:', os.path.basename(voz_ref))

try:
    arquivo
except NameError:
    arquivo = os.path.basename(glob.glob('/content/entrada/*')[0])
base = os.path.splitext(arquivo)[0]
os.makedirs('/content/extras', exist_ok=True)

try:
    from basic_pitch.inference import predict_and_save
    from basic_pitch import ICASSP_2022_MODEL_PATH
    print('Analisando com basic-pitch... 1 a 2 minutos.')
    predict_and_save(
        [voz_ref], '/content/extras',
        save_midi=True, sonify_midi=False, save_model_outputs=False, save_notes=False,
        model_or_model_path=ICASSP_2022_MODEL_PATH
    )
    midis = glob.glob('/content/extras/*.mid*')
    voz_midi = f'/content/extras/{base}_voz.mid'
    if midis:
        os.replace(midis[0], voz_midi)
        print('MIDI de referencia gerado:', os.path.basename(voz_midi))
        try:
            from music21 import converter
            part = converter.parse(voz_midi)
            voz_xml = f'/content/extras/{base}_voz.musicxml'
            part.write('musicxml', fp=voz_xml)
            print('Partitura MusicXML gerada:', os.path.basename(voz_xml))
        except Exception as e:
            print('MusicXML nao gerado (o MIDI ja serve para comparar):', e)
    else:
        print('Nenhum MIDI foi gerado. Verifique se a voz tem conteudo audivel.')
except Exception as e:
    print('A transcricao da voz falhou:', e)
    print('As pistas ja foram baixadas na celula 5 - este extra e opcional.')

# Baixa os extras gerados, se houver.
try:
    from google.colab import files
    for f in glob.glob('/content/extras/*'):
        files.download(f)
except Exception:
    pass